# Red neuronal para predicción de área de incendios forestales
Este notebook muestra los pasos comentados para entrenar y evaluar una red neuronal que predice la variable *area* a partir del dataset **forestfires.csv**.

## 1. Importar librerías
Importamos las librerías necesarias para manipulación de datos, preprocesamiento, modelado y evaluación.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RepeatedKFold

## 2. Cargar el dataset
Leemos el archivo CSV con pandas y separamos la variable objetivo `area`.

In [ ]:
data = pd.read_csv('forestfires.csv')  # Leer datos

# Separar características (X) y variable objetivo (y)
X = data.drop('area', axis=1)  # Todas menos 'area'
raw_y = data['area']           # Área sin transformar

## 3. Transformar la variable objetivo
Aplicamos `log(1 + area)` para controlar la asimetría de la variable *area*.

In [ ]:
y = np.log1p(raw_y)  # Transformación logarítmica

## 4. Normalizar las entradas
Usamos MinMaxScaler para escalar las variables numéricas entre 0 y 1.

In [ ]:
# Definir el escalador
scaler = MinMaxScaler()

# Columnas a normalizar
cols_to_norm = ['X', 'Y', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain']

# Ajustar y transformar
X[cols_to_norm] = scaler.fit_transform(X[cols_to_norm])

## 5. Codificar variables categóricas
Convertimos `month` y `day` en variables dummy (one-hot encoding).

In [ ]:
X = pd.get_dummies(X, columns=['month', 'day'], drop_first=True).astype(float)

## 6. Preparar datos para sklearn
Convertimos `X` y `y` a arrays de NumPy para compatibilidad con scikit-learn.

In [ ]:
X = X.values
y = y.values

## 7. Definir validación cruzada y modelo
- Usamos `RepeatedKFold` con 10 particiones y 30 repeticiones.
- Definimos un `MLPRegressor` con arquitectura 20-10-5, función logística y optimizador SGD.
- Habilitamos *early stopping* para detener entrenamiento si no mejora la validación.

In [ ]:
from sklearn.model_selection import RepeatedKFold
from sklearn.neural_network import MLPRegressor

# Repeated 10-fold CV
rkf = RepeatedKFold(n_splits=10, n_repeats=30, random_state=42)

# Definir la red neuronal
mlp = MLPRegressor(
    hidden_layer_sizes=(20, 10, 5),  # Capas ocultas
    activation='logistic',           # Función de activación sigmoidal
    solver='sgd',                    # Descenso de gradiente estocástico
    max_iter=1000,                   # Máximo de iteraciones
    early_stopping=True,             # Detener si no mejora
    random_state=None                # Inicialización aleatoria
)

## 8. Entrenar y evaluar con CV
Iteramos sobre cada partición, entrenamos el modelo, predecimos y calculamos el MSE.

In [ ]:
# Lists to store metrics for each fold
mse_list = []
mae_list = []
r2_list = []

for train_index, test_index in rkf.split(X):
    # Separar en train y test
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # Entrenar
    mlp.fit(X_train, y_train)
    
    # Predecir
    y_pred = mlp.predict(X_test)
    
    # Compute metrics and store it
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    mse_list.append(mse)
    mae_list.append(mae)
    r2_list.append(r2)

## 9. Resultados finales
Calculamos el MSE promedio sobre todas las particiones y repeticiones.

In [ ]:
# Compute and print the average metrics
average_mse = np.mean(mse_list)
average_mae = np.mean(mae_list)
average_r2 = np.mean(r2_list)

print(f'Average Mean Squared Error: {average_mse:.4f}')
print(f'Average Mean Absolute Error: {average_mae:.4f}')
print(f'Average R^2: {average_r2:.4f}')